# Week 2: Basic Image Transformations with OpenCV

**Computer Vision Fundamentals** — Temasek Poly IIT AAI

*(Student copy — fill in the code where you see **Task** or placeholder comments. Use the teacher copy for full solutions.)*

This notebook covers the most common **geometric** and **photometric** image transformations using OpenCV: resizing, cropping, flipping, rotation, translation, shearing, blurring, and masking.

---
## 3.1 Why Image Transformations?

Image transformations change the **geometry** or **appearance** of an image. They are essential in almost every CV pipeline.

### When do we need them?
- **Preprocessing**: Models often expect a fixed input size (e.g. 224×224). We resize to match.
- **Data augmentation**: Flipping, rotating, and cropping create training-data variety so models generalise better.
- **Correction**: Straightening a tilted document scan, cropping a region of interest (ROI).
- **Noise reduction**: Blurring removes sensor noise before edge detection or thresholding.

### Two families of transforms
| Family | What changes | Examples |
|---|---|---|
| **Geometric** | Pixel *positions* | Resize, crop, flip, rotate, translate, shear |
| **Photometric** | Pixel *values* | Blur, sharpen, brightness/contrast (Week 1) |

We will cover both in this session.

---
## 3.2 Setup: Imports and Loading Images

We reuse the same libraries from Week 1: **OpenCV**, **NumPy**, and **Matplotlib**. We also define a small helper to convert BGR → RGB for display.

**Task:** Write the three import statements and define `bgr_to_rgb(img)` using `cv2.cvtColor`.

In [ ]:
# Import cv2, numpy as np, matplotlib.pyplot as plt

# Define bgr_to_rgb(img) that converts BGR to RGB using cv2.cvtColor


Load the two sample images from the `data` folder.

**Task:** Load `data/cat.jpg` and `data/number_plate.jpg` with `cv2.imread()`. Display them side by side with `plt.subplots(1, 2)`. Print each image's shape and dtype.

In [ ]:
# Load cat.jpg and number_plate.jpg from the data folder using cv2.imread()
# Store them in variables called cat and plate

# Downsize cat to width 400 preserving aspect ratio:
#   h_orig, w_orig = cat.shape[:2]
#   scale_down = 400 / w_orig
#   cat = cv2.resize(cat, (400, int(h_orig * scale_down)))

# Print shape and dtype for each image
# Display both side by side using plt.subplots(1, 2)


---
## 3.3 Resizing

Resizing changes the **width** and **height** of an image. It is probably the most common transformation.

```python
cv2.resize(src, dsize, fx, fy, interpolation)
```

- `dsize` — target size as `(width, height)`. **Note:** this is `(W, H)`, the *opposite* of NumPy's `(H, W)` shape convention!
- `fx`, `fy` — scale factors (use when `dsize=None`).
- `interpolation` — how to compute new pixel values (see below).

### Basic resize and scale factors

**Task:** Resize `cat` to `(200, 150)` and `(800, 600)` using `cv2.resize`. Also use `fx`/`fy` to create half and double versions. Print all shapes and display three in subplots.

In [ ]:
# Resize cat to (200, 150) and (800, 600) using cv2.resize()
# Print the original, small, and large shapes

# Also resize using scale factors: fx=0.5/fy=0.5 and fx=2.0/fy=2.0
# Print half and double shapes

# Display Original, Small, and Large in plt.subplots(1, 3)


### Preserving aspect ratio

If we resize to an *arbitrary* `(W, H)`, the image may be **stretched** or **squashed**. To avoid this, compute the new dimensions from the original **aspect ratio**.

Recipe:
1. Decide a **target width** (or height).
2. Compute `scale = target / original`.
3. Apply the *same* scale to the other dimension.

**Task:** Resize `cat` to width 300 and separately to height 200, preserving the aspect ratio each time. Print the resulting sizes and display all three.

In [ ]:
# Get h, w from cat.shape[:2] and print the aspect ratio
# Resize to target width 300: compute scale, new_h, call cv2.resize
# Resize to target height 200: compute scale_h, new_w, call cv2.resize
# Print both resulting sizes and display in plt.subplots(1, 3)


### Interpolation methods

When we resize, OpenCV must compute pixel values at *new* positions. **Interpolation** is the method used.

| Method | Flag | Best for | Speed |
|---|---|---|---|
| Nearest-neighbour | `cv2.INTER_NEAREST` | Fast preview, pixel art | Fastest |
| Bilinear | `cv2.INTER_LINEAR` | General upscaling (default) | Fast |
| Bicubic | `cv2.INTER_CUBIC` | High-quality upscaling | Medium |
| Area-based | `cv2.INTER_AREA` | **Downscaling** (avoids aliasing) | Medium |

**Rule of thumb:** Use `INTER_AREA` when shrinking, `INTER_LINEAR` or `INTER_CUBIC` when enlarging.

**Task:** Shrink `cat` to 40×40, then upscale to 400×400 using each method. Display all four in subplots.

In [ ]:
# Shrink cat to (40, 40) with cv2.resize
# Upscale back to (400, 400) using each interpolation method:
#   cv2.INTER_NEAREST, cv2.INTER_LINEAR, cv2.INTER_CUBIC, cv2.INTER_AREA
# Display all four in plt.subplots(1, 4) with titles


---
## 3.4 Cropping (Region of Interest)

Cropping extracts a rectangular **region of interest (ROI)** from the image. In OpenCV+NumPy, cropping is just **array slicing**:

```python
roi = img[y1:y2, x1:x2]   # rows y1..y2-1, columns x1..x2-1
```

Remember: NumPy indexing is `[row, col]` = `[y, x]`.

**Task:** Crop the centre quarter of the cat image and display it next to the original.

In [ ]:
# Get h, w from cat.shape[:2]
# Compute y1, y2 = h//4, 3*h//4 and x1, x2 = w//4, 3*w//4
# Crop with cat[y1:y2, x1:x2] and store in center_crop
# Print both shapes and display in plt.subplots(1, 2)


### Practical: Crop the license plate

A common CV task is to **isolate** a region for further processing (e.g. OCR on a license plate). Here we manually crop the plate text from `number_plate.jpg`.

> In a real system, you would *detect* the plate region automatically (Week 2.2 covers contour detection). For now, we eyeball the coordinates.

**Task:** Crop the yellow plate region from `plate`. Use percentage-based coordinates (e.g. `int(ph * 0.15)`) so the code adapts to any image size.

In [ ]:
# Get ph, pw from plate.shape[:2]
# Define plate_y1, plate_y2, plate_x1, plate_x2 using percentage-based coordinates
# Crop with plate[plate_y1:plate_y2, plate_x1:plate_x2]
# Display original and cropped in plt.subplots(1, 2)


---
## 3.5 Flipping

Flipping mirrors the image along an axis.

```python
cv2.flip(src, flipCode)
```

| `flipCode` | Effect |
|---|---|
| `1` | Horizontal (mirror left / right) |
| `0` | Vertical (mirror top / bottom) |
| `-1` | Both (rotate 180 degrees) |

Flipping is used heavily in **data augmentation** — a horizontally flipped cat is still a cat, so it doubles your training data for free.

**Task:** Create horizontal, vertical, and both-flipped versions of `cat`. Display all four (including original) in subplots.

In [ ]:
# Create flip_h (horizontal, code=1), flip_v (vertical, code=0), flip_b (both, code=-1)
# Display all four (original + three flips) in plt.subplots(1, 4)


---
## 3.6 Rotation

### Simple 90 / 180 / 270 degree rotation

For exact multiples of 90 degrees, use `cv2.rotate()` — it is fast and lossless (no interpolation needed).

| Flag | Rotation |
|---|---|
| `cv2.ROTATE_90_CLOCKWISE` | 90 deg CW |
| `cv2.ROTATE_180` | 180 deg |
| `cv2.ROTATE_90_COUNTERCLOCKWISE` | 90 deg CCW (= 270 deg CW) |

**Task:** Create 90, 180, and 270 degree rotations of `cat` and display all four.

In [ ]:
# Use cv2.rotate() with the three flags to create rot_90, rot_180, rot_270
# Display all four (original + three rotations) in plt.subplots(1, 4)


### Arbitrary-angle rotation with `cv2.getRotationMatrix2D`

For any angle, we need a **2×3 affine transformation matrix**:

```python
M = cv2.getRotationMatrix2D(center, angle, scale)
rotated = cv2.warpAffine(src, M, (width, height))
```

- `center` — the point to rotate around, usually the image centre.
- `angle` — rotation in degrees (positive = counter-clockwise).
- `scale` — optional scaling factor (1.0 = no scaling).

**Gotcha:** `warpAffine` uses the *original* canvas size by default, so corners may be **clipped**.

**Task:** Rotate `cat` by 30 degrees (scale 1.0) and 45 degrees (scale 0.7). Print the rotation matrix for 30 degrees. Display original and both rotations.

In [ ]:
# Get h, w and compute center = (w // 2, h // 2)
# Use cv2.getRotationMatrix2D(center, 30, 1.0) to get M_30
# Apply with cv2.warpAffine(cat, M_30, (w, h)) -> rotated_30
# Do the same for 45 degrees with scale=0.7 -> rotated_45
# Print M_30 and display all three in plt.subplots(1, 3)


### Rotation without clipping

To keep the entire image visible after rotation, we **expand the canvas** to fit the new bounding box, and adjust the translation in the rotation matrix.

Steps:
1. Compute the rotation matrix.
2. Calculate the new bounding-box width and height using the absolute cosine and sine components.
3. Adjust the translation (last column of the matrix) so the centre stays centred.
4. Use the new dimensions in `warpAffine`.

**Task:** Complete the `rotate_full` function below and test it at 30 and 45 degrees.

In [ ]:
def rotate_full(img, angle):
    """Rotate an image by angle degrees without clipping."""
    h, w = img.shape[:2]
    center = (w / 2, h / 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)

    # TODO: compute cos_a, sin_a from M[0, 0] and M[0, 1]
    # TODO: compute new_w and new_h for the expanded bounding box
    # TODO: adjust M[0, 2] and M[1, 2] to keep the image centred
    # TODO: return cv2.warpAffine with (new_w, new_h)
    pass

# Test with 30 and 45 degrees, display in plt.subplots(1, 3)


---
## 3.7 Translation (Shifting)

Translation moves every pixel by `(tx, ty)`. We build a **2×3 affine matrix** manually:

```
M = [[1, 0, tx],
     [0, 1, ty]]
```

- `tx > 0` shifts **right**, `tx < 0` shifts **left**.
- `ty > 0` shifts **down**, `ty < 0` shifts **up**.

Pixels that move outside the canvas are discarded; the empty area is filled with black (0) by default.

**Task:** Translate `cat` by (100, 50) right-down and (-80, -60) left-up. Display all three.

In [ ]:
# Build a 2x3 translation matrix with np.float32 for (tx=100, ty=50)
# Apply with cv2.warpAffine(cat, M, (w, h))
# Do the same for (-80, -60)
# Display original and both translations in plt.subplots(1, 3)


---
## 3.8 Shearing

A **shear** transformation slants the image along one axis. It is an *affine* transformation described by a 2×3 matrix.

**Horizontal shear** (`shx`):
```
M = [[1, shx, 0],
     [0,   1, 0]]
```
Each row is shifted proportionally to its y-position.

**Vertical shear** (`shy`):
```
M = [[  1, 0, 0],
     [shy, 1, 0]]
```
Each column is shifted proportionally to its x-position.

We typically expand the canvas to fit the result.

**Task:** Apply horizontal shear (shx=0.3) and vertical shear (shy=0.3) to `cat`. Expand the canvas so nothing is clipped. Display all three.

In [ ]:
# Build horizontal shear matrix with shx=0.3, expand canvas width
# Build vertical shear matrix with shy=0.3, expand canvas height
# Apply both with cv2.warpAffine and display in plt.subplots(1, 3)


---
## 3.9 Blurring and Smoothing

Blurring is a **photometric** transformation that reduces noise and fine detail. It replaces each pixel with a weighted average of its neighbours.

### Why blur?
- **Noise reduction**: Cameras introduce noise; blurring smooths it out.
- **Preprocessing**: Many algorithms (edge detection, thresholding) work better on slightly blurred images.
- **Privacy**: Blur faces or license plates before publishing.

### Four common methods

| Method | Function | Kernel | Best for |
|---|---|---|---|
| Average (box) | `cv2.blur(img, (k,k))` | All weights equal | Simple, fast |
| Gaussian | `cv2.GaussianBlur(img, (k,k), sigmaX)` | Bell-curve weights | General noise |
| Median | `cv2.medianBlur(img, k)` | Median of neighbours | Salt-and-pepper noise |
| Bilateral | `cv2.bilateralFilter(img, d, sigmaColor, sigmaSpace)` | Edge-aware | Smoothing *while* keeping edges |

The **kernel size** `k` must be a **positive odd integer** (3, 5, 7, ...). Larger kernel = more blur.

**Task:** Apply all four blur methods to `cat` with kernel size 15. Display all five (original + 4 blurs) in subplots.

In [ ]:
# Set k = 15
# Apply cv2.blur, cv2.GaussianBlur, cv2.medianBlur, cv2.bilateralFilter
# Display all five (original + 4 blurs) in plt.subplots(1, 5)


---
## 3.10 Masking and Selective Blur

A **mask** is a single-channel (grayscale) image where:
- **White (255)** = keep the pixel
- **Black (0)** = ignore / hide the pixel

We combine masks with the original image using **bitwise operations**:
- `cv2.bitwise_and(src, src, mask=mask)` — keeps pixels where mask is white.

### Creating masks
- **Rectangular**: set a slice of a zeros array to 255.
- **Circular**: use `cv2.circle()` on a zeros array.

### Practical use: selective blur

A common use of masks is **selective blurring** — keeping a subject sharp while blurring the background (a "bokeh" effect). The idea:
1. Create a circular mask around the subject.
2. Blur the entire image.
3. **Composite**: where mask is white → use the original (sharp); where mask is black → use the blurred version.

We can composite using `np.where()` with a 3-channel version of the mask.

**Task:** Create a circular mask centred on the cat. Blur the full image with a large Gaussian kernel. Use `np.where` to composite the sharp cat inside the circle with the blurred background outside.

In [ ]:
# Create a circular mask: np.zeros((h, w), dtype=np.uint8)
# Use cv2.circle(..., 255, -1) to draw a filled white circle at the centre

# Blur the full cat image with cv2.GaussianBlur using a (31, 31) kernel

# Composite using np.where:
#   1. Expand mask to 3 channels: cv2.cvtColor(mask_circle, cv2.COLOR_GRAY2BGR)
#   2. np.where(mask_3ch == 255, cat, blurred)  -> sharp inside, blurred outside

# Display Original, Mask, Fully Blurred, and Selective Blur in plt.subplots(1, 4)


---
## 3.11 In-class Application: License Plate Preprocessing

Let's combine several transformations into a **mini pipeline** — the kind of preprocessing you would do before feeding a license plate to an OCR model:

1. **Crop** the plate region.
2. **Resize** to a standard width (preserving aspect ratio).
3. **Convert to grayscale**.
4. **Blur** to reduce noise.

This previews the full segmentation pipeline we will build in Week 2.2.

**Task:** Implement the four-step pipeline and display each step in subplots.

In [ ]:
# Step 1: Crop the plate region from plate using slicing
# Step 2: Resize to width=400, preserving aspect ratio
# Step 3: Convert to grayscale with cv2.cvtColor
# Step 4: Apply Gaussian blur with kernel (5, 5)
# Display all four steps in plt.subplots(1, 4)


---
## 3.12 Summary

| Transformation | Key function(s) | Notes |
|---|---|---|
| **Resize** | `cv2.resize(img, (W,H))` | Watch out for `(W,H)` vs NumPy `(H,W)`. Use `INTER_AREA` for downscaling. |
| **Crop** | `img[y1:y2, x1:x2]` | Pure NumPy slicing. |
| **Flip** | `cv2.flip(img, code)` | `1`=horiz, `0`=vert, `-1`=both. |
| **Rotate (90 deg)** | `cv2.rotate(img, flag)` | Fast, lossless for 90/180/270. |
| **Rotate (any)** | `getRotationMatrix2D` + `warpAffine` | Expand canvas to avoid clipping. |
| **Translate** | `warpAffine` with `[[1,0,tx],[0,1,ty]]` | Shifts the image. |
| **Shear** | `warpAffine` with shear matrix | Slants along one axis. |
| **Blur** | `blur`, `GaussianBlur`, `medianBlur`, `bilateralFilter` | Odd kernel sizes. Bilateral preserves edges. |
| **Mask** | `np.where(mask, original, blurred)` | Use masks for selective effects (e.g. blur background, keep subject sharp). |

**Next week (2.2):** We build on these skills — thresholding, edge detection, contour detection, and perspective correction for a full license-plate recognition pipeline.